# Mamba-Attention Hybrid Training on Colab

Trains the hybrid Mamba-2 + sparse attention LM on a Colab GPU.

- Checkpoints saved to Google Drive (survive session resets)
- Auto-resumes from latest checkpoint
- Falls back to pure PyTorch Mamba if CUDA kernel won't compile

In [ ]:
# @title 1. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_DIR = '/content/drive/MyDrive/mamba-attention'
CKPT_DIR = f'{DRIVE_DIR}/checkpoints'
os.makedirs(CKPT_DIR, exist_ok=True)
print(f'Checkpoints will be saved to: {CKPT_DIR}')

In [ ]:
# @title 2. Clone project from GitHub
PROJECT_DIR = '/content/mamba-attention-hybrid'

if not os.path.exists(PROJECT_DIR):
    !git clone https://github.com/Francis261/NamAi.git {PROJECT_DIR}

%cd {PROJECT_DIR}
print(f'\nWorking directory: {os.getcwd()}')

In [ ]:
# @title 3. Install dependencies
import torch
print(f'GPU: {torch.cuda.get_device_name()}')
print(f'Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
print(f'CUDA: {torch.version.cuda}  PyTorch: {torch.__version__}')

# Core dependencies (always needed)
!pip install -q transformers datasets safetensors wandb tiktoken

# Try to install Mamba CUDA kernel (compiles from source, can fail)
mamba_available = False
try:
    !pip install -q causal-conv1d>=1.4.0
    !pip install -q mamba-ssm>=2.2.0
    from mamba_ssm import Mamba2
    mamba_available = True
    print('\nMamba CUDA kernel: INSTALLED')
except Exception as e:
    print(f'\nMamba CUDA kernel: FAILED ({e})')
    print('Using pure PyTorch fallback (slower but works)')

# Verify model can be created
from src.model.config import ModelConfig
from src.model.model import MambaAttentionLM
cfg = ModelConfig.from_yaml('configs/tiny_cpu.yaml')
m = MambaAttentionLM(cfg).to('cuda')
print(f'Model creation: OK ({sum(p.numel() for p in m.parameters()):,} params)')
del m, cfg

In [ ]:
# @title 4. Configure training
# @markdown ---
# @markdown **Model**
model_config = 'configs/350m.yaml' # @param ['configs/350m.yaml', 'configs/760m.yaml', 'configs/1.2b.yaml', 'configs/tiny_cpu.yaml']

# @markdown **Data**
dataset_name = 'c4' # @param ['c4', 'wikitext-103-raw-v1']

# @markdown **Training hyperparameters**
batch_size = 4 # @param {type:'integer'}
grad_accum = 2 # @param {type:'integer'}
max_steps = 50000 # @param {type:'integer'}
learning_rate = 3e-4 # @param {type:'number'}
warmup_steps = 2000 # @param {type:'integer'}
save_interval = 500 # @param {type:'integer'}
log_interval = 10 # @param {type:'integer'}

# @markdown **Weights & Biases (optional)**
use_wandb = False # @param {type:'boolean'}
wandb_key = '' # @param {type:'string'}

print(f'Model: {model_config}')
print(f'Batch: {batch_size} x {grad_accum} = {batch_size*grad_accum} effective')
print(f'Steps: {max_steps}  LR: {learning_rate}  Warmup: {warmup_steps}')

In [ ]:
# @title 5. Find latest checkpoint (for resume)
resume_path = None
if os.path.exists(CKPT_DIR):
    step_dirs = [d for d in os.listdir(CKPT_DIR) if d.startswith('step_')]
    if step_dirs:
        steps = [int(d.replace('step_', '')) for d in step_dirs]
        best_idx = steps.index(max(steps))
        resume_path = os.path.join(CKPT_DIR, step_dirs[best_idx])
        print(f'Resuming from: {resume_path} (step {max(steps)})')
    else:
        print('No checkpoints found. Starting fresh.')
else:
    print('No checkpoints found. Starting fresh.')

In [ ]:
# @title 6. Start training
%cd {PROJECT_DIR}

import sys, math, time
sys.path.insert(0, PROJECT_DIR)

from src.model.config import ModelConfig
from src.model.model import MambaAttentionLM
from src.training.optimizer import configure_optimizer, get_cosine_schedule
from src.training.data import create_dataloader
from src.model.save_load import save_checkpoint, load_checkpoint

config = ModelConfig.from_yaml(model_config)
device = torch.device('cuda')
torch.manual_seed(42)

model = MambaAttentionLM(config).to(device)
total = sum(p.numel() for p in model.parameters())
print(f'Model: {total:,} params')

optimizer = configure_optimizer(model, weight_decay=0.1, lr=learning_rate)
scheduler = get_cosine_schedule(optimizer, warmup_steps, max_steps, 1e-5)

step = 0
tokens_seen = 0
if resume_path:
    try:
        model, extra = load_checkpoint(resume_path, device=device)
        if 'optimizer_state_dict' in extra:
            optimizer.load_state_dict(extra['optimizer_state_dict'])
        step = extra.get('step', 0)
        tokens_seen = extra.get('tokens_seen', 0)
        for _ in range(step):
            scheduler.step()
        print(f'Resumed at step {step}, tokens {tokens_seen:,}')
    except Exception as e:
        print(f'Resume failed: {e}. Starting fresh.')

if use_wandb and wandb_key:
    import wandb
    wandb.login(key=wandb_key)
    wandb.init(project='mamba-attention-hybrid', name='colab')

dataloader = create_dataloader(
    batch_size=batch_size,
    max_seq_len=config.max_seq_len,
    dataset_name=dataset_name,
    split='train',
)

scaler = torch.amp.GradScaler('cuda', enabled=True)
model.train()

print('\nTraining... (Ctrl+M to stop)')
while step < max_steps:
    for batch in dataloader:
        if step >= max_steps:
            break

        input_ids = batch['input_ids'].to(device)
        labels = batch['labels'].to(device)

        with torch.amp.autocast('cuda', dtype=torch.bfloat16, enabled=True):
            _, loss = model(input_ids, labels)

        scaler.scale(loss).backward()

        if (step + 1) % grad_accum == 0:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()
            scheduler.step()

        tokens_seen += input_ids.numel()

        if step % log_interval == 0:
            ppl = math.exp(min(loss.item(), 20))
            lr_now = scheduler.get_last_lr()[0]
            print(f'step={step:>5d} | loss={loss.item():.4f} | ppl={ppl:.2f} | '
                  f'lr={lr_now:.2e} | tok={tokens_seen:,}')
            if use_wandb and wandb_key:
                wandb.log({'loss': loss.item(), 'perplexity': ppl,
                          'step': step, 'tokens': tokens_seen})

        if step > 0 and step % save_interval == 0:
            save_checkpoint(CKPT_DIR, f'step_{step}', model, config,
                step=step, optimizer_state_dict=optimizer.state_dict(),
                loss=loss.item(), tokens_seen=tokens_seen)

        step += 1

# Final save
save_checkpoint(CKPT_DIR, 'final', model, config,
    step=step, loss=loss.item(), tokens_seen=tokens_seen)
print('\nTraining complete!')

In [ ]:
# @title 7. Generate sample text
from src.model.tokenizer import get_tokenizer

tokenizer = get_tokenizer()
prompt = 'The future of AI is'
ids = tokenizer.encode(prompt)
input_ids = torch.tensor(ids, dtype=torch.long, device=device).unsqueeze(0)

model.eval()
with torch.no_grad():
    for _ in range(50):
        logits, _ = model(input_ids)
        logits = logits[:, -1, :] / 0.8
        probs = torch.softmax(logits, dim=-1)
        next_id = torch.multinomial(probs, num_samples=1)
        input_ids = torch.cat([input_ids, next_id], dim=1)

print('Prompt:', prompt)
print('Output:', tokenizer.decode(input_ids[0].tolist()))

## Resume on next session

Re-run cells in order:
1. Mount Drive
2. Clone (will skip if already exists)
3. Install deps (will skip if already installed)
4. Configure
5. Find checkpoint (auto-finds latest)
6. Start training (resumes from checkpoint)